# Week 3 — Researcher–model exchange about mutual aid

**Research question:** When, if at all, does mutual aid among tenants contribute to political solidarity?

Read eight synthetic interview excerpts, write a first memo, question the model twice using evidence, and make your own provisional judgment. The model is a participant in the analytic exchange, not the author of the final claim.

**Python introduced:** text strings, lists of source dictionaries, `input(...)`, `if/else`, JSON text and dictionaries, and how a sequence of messages preserves previous turns.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session03/session03_qualitative_interpretation.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell first. In Colab it installs missing SDKs and downloads the public course repository. Locally it checks the course environment. This setup code is supplied and is not assessed; the subsequent cells show every model call in full.

Colab uses OpenRouter. To use Ollama, open this notebook in local JupyterLab or VS Code with the course environment. The full setup instructions are in `docs/ENVIRONMENT_SETUP.md` and the book's computing chapter.

In [ ]:
SESSION = "session03"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## The material and route

The eight excerpts are instructor-authored. `T01_A` and `T01_B` are two moments in one fictional tenant's account. `T02_A`, `T03_A` and `T05_A` make a universal conversion story difficult; `T04_A` and `T07_A` show collective action more directly. `T06_A` follows a different interviewer prompt, which is recorded in `context`.

**Input:** the route and eight dictionaries. **Output:** a research-question string, a list of source records and their IDs. Colab selects OpenRouter; local work defaults to Ollama. The key prompt, when needed, is hidden. Do not record the key.

In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # local default
if ROUTE not in ("ollama", "openrouter"):
    raise ValueError("ROUTE must be 'ollama' or 'openrouter'.")
if ROUTE == "openrouter" and not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

research_question = (
    "When, if at all, does mutual aid among tenants contribute "
    "to political solidarity?"
)
excerpts = [
    {
        "id": "T01_A",
        "text": "When the rent went up, grocery deliveries got us through the last week of the month. At that point I did not know who organized them.",
    },
    {
        "id": "T01_B",
        "text": "After a few months of taking turns with childcare, I started going to the tenants' meetings. The first time I spoke was about the broken elevator.",
    },
    {
        "id": "T02_A",
        "text": "I was grateful for the emergency fund, but I kept it separate from politics. I did not attend meetings and I still do not think of myself as an organizer.",
    },
    {
        "id": "T03_A",
        "text": "Once the delivery rota became a requirement, it stopped feeling like help. I missed two shifts, people kept messaging me, and I left the group.",
    },
    {
        "id": "T04_A",
        "text": "Fixing things together meant we already trusted one another when the eviction notices arrived. Six of us wrote the petition and delivered it together.",
    },
    {
        "id": "T05_A",
        "text": "The organizers call it solidarity. For me it was mostly a way to make it to payday. I did not want a political identity attached to it.",
    },
    {
        "id": "T06_A",
        "text": "I learned people's names and felt less alone. I would not say it made me political, but I was more willing to ask what others needed.",
        "context": "An AI-moderated follow-up asked whether deliveries changed relationships.",
    },
    {
        "id": "T07_A",
        "text": "The food deliveries ended for me, but the meetings were where we compared what the landlord had told each floor. I stayed because acting separately was not working.",
    },
]
print("Research question:", research_question)
for excerpt in excerpts:
    print(excerpt["id"], "—", excerpt["text"])
print("T06_A interviewer context:", excerpts[6]["context"])

## Write before seeing a suggestion

`input(...)` pauses and stores what you type as a **string**. Write your own one-sentence interpretation, not the printed example. The model will not see this first memo. That lets you later identify what the model made you notice or overlook.

In [ ]:
print("Example first memo: Help may build relationships, but receiving aid alone is not collective action.")
first_memo = input("Your one-sentence first reading: ")
if not first_memo.strip():
    raise ValueError("Write a first memo before asking the model. Then rerun this cell.")
print("Saved before the model:", first_memo)

## Round 1 — ask for a provisional pattern

`json.dumps(excerpts)` turns the Python list into text for the prompt. `messages_1` is a list with one user message. The `if/else` chooses the hosted or local call, and both branches place returned text in `raw_1`. Read the raw JSON before parsing it. The model may cite a source and still misread it.

In [ ]:
prompt_1 = (
    "Research question: " + research_question + "\n"
    "For this exercise, political solidarity means tenants identifying a shared "
    "problem and acting together; receiving aid alone does not establish it. "
    "Suggest one provisional pattern and cite one supplied excerpt ID. "
    "Return a JSON object with exactly theme, evidence_id, and "
    "question_for_researcher. These are synthetic interview excerpts: "
    + json.dumps(excerpts)
)
messages_1 = [{"role": "user", "content": prompt_1}]
print("Round 1 sends", len(excerpts), "labelled excerpts in one user message.")

**The call:** The `if/else` selects one route. OpenRouter sends the message to a hosted service; Ollama sends it to the local model. Both assign returned JSON text to `raw_1`. `temperature=0` reduces variation but does not validate the theme. The raw text is printed before parsing.

In [ ]:
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response_1 = client.chat.send(
            model=HOSTED_MODEL,
            messages=messages_1,
            temperature=0,
            max_tokens=250,
            response_format={"type": "json_object"},
        )
    raw_1 = response_1.choices[0].message.content
else:
    response_1 = ollama.chat(think=False,
        model=LOCAL_MODEL,
        messages=messages_1,
        format="json",
        options={"temperature": 0, "num_predict": 250},
    )
    raw_1 = response_1.message.content
print("Round 1 raw output:", raw_1)

## Return to the cited passage

`json.loads(raw_1)` converts JSON text into a Python value. `isinstance(..., dict)` checks that it is a dictionary, not a list or number. `all(...)` checks that all three requested fields have values; otherwise the code stops with a readable error. `.get(...)` retrieves a named value or `None` if absent. The short `for` loop searches the eight original records for the cited ID. If the ID is absent, the code warns you so you can challenge it in your next reply. **Output:** the model's theme, cited ID, full cited record, its question and your earlier memo. Finding an ID does not prove that the passage supports the theme. If the model returns invalid JSON, rerun only that model-call cell and this source-check cell.

In [ ]:
proposal_1 = json.loads(raw_1)
if not isinstance(proposal_1, dict) or not all(
    proposal_1.get(field) for field in ("theme", "evidence_id", "question_for_researcher")
):
    raise ValueError("Round 1 needs theme, evidence_id and question_for_researcher. Rerun the model call.")
print("Round 1 fields:", list(proposal_1))
print("Round 1 theme:", proposal_1.get("theme"))
print("Round 1 cited ID:", proposal_1.get("evidence_id"))
cited_1 = None
for excerpt in excerpts:
    if excerpt["id"] == proposal_1.get("evidence_id"):
        cited_1 = excerpt
print("Round 1 cited passage:", cited_1)
if cited_1 is None:
    print("The model cited an ID not in the eight excerpts. Challenge that in your reply.")
print("Round 1 model question:", proposal_1.get("question_for_researcher"))
print("Your earlier memo:", first_memo)
print("Compare T02_A, T03_A and T05_A before accepting a general claim.")

## Round 2 — challenge the proposal

Type your own reply with at least one source ID. You might point to `T02_A`, where help remained separate from politics, or `T03_A`, where an obligation drove someone away. `messages_2` adds the model's first answer and your reply to the earlier message list; the model receives that visible history. The same route makes a second call. Compare the new theme with both the first answer and the source.

In [ ]:
print(
    "Example reply: T02_A accepted help but rejected political involvement; "
    "T03_A left when help became obligatory. How does your theme handle both?"
)
feedback_1 = input("Your reply to the model, naming at least one excerpt ID: ")
if not feedback_1.strip():
    raise ValueError("Write a reply naming a countercase before the second call.")
messages_2 = messages_1 + [
    {"role": "assistant", "content": raw_1},
    {
        "role": "user",
        "content": (
            feedback_1
            + " Reconsider your theme using the supplied excerpts. Do not simply "
            "agree with me; explain any evidence that resists my challenge. "
            "Return JSON with exactly theme, evidence_id, and "
            "question_for_researcher."
        ),
    },
]
print("Round 2 new researcher input:", feedback_1)

**The second call:** It receives `messages_2`, which includes the first prompt, the first model answer and your reply. The route-specific branches remain visible. The output is raw JSON text in `raw_2`.

In [ ]:
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response_2 = client.chat.send(
            model=HOSTED_MODEL,
            messages=messages_2,
            temperature=0,
            max_tokens=250,
            response_format={"type": "json_object"},
        )
    raw_2 = response_2.choices[0].message.content
else:
    response_2 = ollama.chat(think=False,
        model=LOCAL_MODEL,
        messages=messages_2,
        format="json",
        options={"temperature": 0, "num_predict": 250},
    )
    raw_2 = response_2.message.content
print("Round 2 raw output:", raw_2)

**Check the revision:** `json.loads` makes a dictionary. Read its theme, find the cited source and decide whether it addressed your countercase or merely became agreeable.

In [ ]:
proposal_2 = json.loads(raw_2)
if not isinstance(proposal_2, dict) or not all(
    proposal_2.get(field) for field in ("theme", "evidence_id", "question_for_researcher")
):
    raise ValueError("Round 2 needs theme, evidence_id and question_for_researcher. Rerun the model call.")
print("Round 2 theme:", proposal_2.get("theme"))
cited_2 = None
for excerpt in excerpts:
    if excerpt["id"] == proposal_2.get("evidence_id"):
        cited_2 = excerpt
print("Round 2 cited passage:", cited_2)
if cited_2 is None:
    print("The model cited an ID not in the eight excerpts. Challenge that in your reply.")
print("Round 2 model question:", proposal_2.get("question_for_researcher"))

## Round 3 — ask what the evidence really establishes

Now question a possible mechanism. `T04_A` reports trust and a jointly delivered petition, but a participant's account does not by itself prove that trust caused collective action. Your second reply and the model's previous answer form `messages_3`. Check whether the third answer distinguishes reported events, inferred explanations and the limits of this tiny corpus.

In [ ]:
print(
    "Example reply: T04_A mentions trust and a petition. What is observed, "
    "and what is only an interpretation about why tenants acted together?"
)
feedback_2 = input("Your second reply, naming an excerpt and a remaining question: ")
if not feedback_2.strip():
    raise ValueError("Write a second reply about evidence and explanation before the third call.")
messages_3 = messages_2 + [
    {"role": "assistant", "content": raw_2},
    {
        "role": "user",
        "content": (
            feedback_2
            + " Distinguish reported events from an inferred mechanism. "
            "Do not claim these few interviews establish a causal effect. "
            "Keep the theme about mutual aid and solidarity; put remaining "
            "methodological uncertainty in question_for_researcher. "
            "Return JSON with exactly theme, evidence_id, and "
            "question_for_researcher."
        ),
    },
]
print("Round 3 new researcher input:", feedback_2)

**The third call:** It receives the complete visible message history, including your second question. The output is raw text in `raw_3`. This is another request, not a hidden continuation.

In [ ]:
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response_3 = client.chat.send(
            model=HOSTED_MODEL,
            messages=messages_3,
            temperature=0,
            max_tokens=250,
            response_format={"type": "json_object"},
        )
    raw_3 = response_3.choices[0].message.content
else:
    response_3 = ollama.chat(think=False,
        model=LOCAL_MODEL,
        messages=messages_3,
        format="json",
        options={"temperature": 0, "num_predict": 250},
    )
    raw_3 = response_3.message.content
print("Round 3 raw output:", raw_3)

**Check the third answer:** Read the parsed theme and the cited original passage. Separate a reported event from a causal explanation the interviews cannot establish.

In [ ]:
proposal_3 = json.loads(raw_3)
if not isinstance(proposal_3, dict) or not all(
    proposal_3.get(field) for field in ("theme", "evidence_id", "question_for_researcher")
):
    raise ValueError("Round 3 needs theme, evidence_id and question_for_researcher. Rerun the model call.")
print("Round 3 theme:", proposal_3.get("theme"))
cited_3 = None
for excerpt in excerpts:
    if excerpt["id"] == proposal_3.get("evidence_id"):
        cited_3 = excerpt
print("Round 3 cited passage:", cited_3)
if cited_3 is None:
    print("The model cited an ID not in the eight excerpts. Treat that as a source-check failure.")
print("Round 3 model question:", proposal_3.get("question_for_researcher"))

## Your conclusion and the saved exchange

`input(...)` asks for your own provisional answer, including a countercase and a limit. Do not merely copy the model's latest wording. `analysis_record` keeps the research question, original excerpts, route, first memo, all three requests and raw returns, your two replies and final memo. This is an inspectable account of how the interpretation developed, not proof that it is correct.

In [ ]:
print("The model's three themes:", proposal_1.get("theme"), "|", proposal_2.get("theme"), "|", proposal_3.get("theme"))
print("Your independent first memo:", first_memo)
final_memo = input(
    "Your provisional answer to the research question, with a countercase and limit: "
)
if not final_memo.strip():
    raise ValueError("Write your own provisional answer before completing the exercise.")
if ROUTE == "openrouter":
    selected_model = HOSTED_MODEL
else:
    selected_model = LOCAL_MODEL

**Save the exchange:** The dictionary below combines the source list, route, three requests and raw replies, your challenges and final memo. It stores an audit trail; it does not judge the interpretation for you.

In [ ]:
analysis_record = {
    "research_question": research_question,
    "first_memo": first_memo,
    "route": ROUTE,
    "model": selected_model,
    "excerpts": excerpts,
    "round_1": {"messages": messages_1, "raw": raw_1, "proposal": proposal_1, "cited": cited_1},
    "round_2": {"feedback": feedback_1, "messages": messages_2, "raw": raw_2, "proposal": proposal_2, "cited": cited_2},
    "round_3": {"feedback": feedback_2, "messages": messages_3, "raw": raw_3, "proposal": proposal_3, "cited": cited_3},
    "final_memo": final_memo,
}
print("Final researcher memo:", analysis_record["final_memo"])
print("Saved rounds:", list(analysis_record))

## Completion recording

Show the initial memo, each researcher reply, the three raw model returns, the cited sources and your final memo. Explain the input and output of each operation. A strong recording notices at least one point where the model overstates what the excerpts establish. Use only these synthetic records and keep your API key off screen.